Sentiment Analysis with an LSTM (Tanh)

In [6]:
# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [7]:
# CELL 2 — Load IMDb dataset
# pip install datasets
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
train_data = ds["train"]
test_data = ds["test"]

print(f"Train samples: {len(train_data)} | Test samples: {len(test_data)}")
print("Example:", train_data[0]["text"][:200], "| Label:", train_data[0]["label"])

Train samples: 25000 | Test samples: 25000
Example: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev | Label: 0


In [8]:
# CELL 3 — Build vocabulary
def tokenize(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)         # remove HTML line breaks
    text = re.sub(r"[^a-z0-9' ]", " ", text)        # strip punctuation
    return text.split()

MAX_VOCAB_SIZE = 10000
counter = Counter()

# Build vocab from a subset for speed
subset_for_vocab = train_data.select(range(5000))
for item in subset_for_vocab:
    counter.update(tokenize(item["text"]))

most_common = counter.most_common(MAX_VOCAB_SIZE - 2)  # reserve <pad>, <unk>
vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

Vocab size: 10000


In [9]:
# CELL 4 — Encode text to sequences of token ids
MAX_LEN = 200

def encode(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

class IMDbDataset(Dataset):
    def __init__(self, hf_data, vocab, max_len=MAX_LEN):
        self.data = hf_data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        ids = encode(item["text"], self.vocab, self.max_len)
        label = item["label"]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Use manageable subsets for quick training
train_subset = train_data.shuffle(seed=42).select(range(5000))
test_subset = test_data.shuffle(seed=42).select(range(1000))

train_dataset = IMDbDataset(train_subset, vocab)
test_dataset = IMDbDataset(test_subset, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")

Train samples: 5000 | Test samples: 1000


In [10]:
# CELL 5 — Model definition
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        # nn.LSTM uses tanh internally for the cell/hidden state activation,
        # combined with sigmoid gates — this is standard and not swapped out
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)                  # [batch, seq_len, embed_dim]
        output, (hidden, cell) = self.lstm(embedded)   # hidden: [1, batch, hidden_dim]
        final_hidden = hidden.squeeze(0)               # [batch, hidden_dim]
        logits = self.fc(final_hidden)                 # raw logits
        return logits

model = SentimentLSTM(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

SentimentLSTM(
  (embedding): Embedding(10000, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [11]:
# CELL 6 — Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(sequences)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

Epoch 1/5 — Loss: 0.6945 | Train Acc: 50.56%
Epoch 2/5 — Loss: 0.6761 | Train Acc: 55.02%
Epoch 3/5 — Loss: 0.6309 | Train Acc: 60.58%
Epoch 4/5 — Loss: 0.5538 | Train Acc: 65.82%
Epoch 5/5 — Loss: 0.4657 | Train Acc: 70.68%


In [12]:
# CELL 7 — Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for sequences, labels in test_loader:
        sequences, labels = sequences.to(device), labels.to(device)
        logits = model(sequences)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 52.70%


In [13]:
# CELL 8 — Try it on custom reviews
def predict_sentiment(text, model, vocab):
    model.eval()
    ids = encode(text, vocab)
    tensor = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
    label = "Positive" if pred == 1 else "Negative"
    confidence = probs[0][pred].item()
    return label, confidence

sample_reviews = [
    "I absolutely loved this film, the acting was superb.",
    "This was a complete waste of time, terrible plot.",
    "An average movie, nothing special but not bad either."
]

for review in sample_reviews:
    label, conf = predict_sentiment(review, model, vocab)
    print(f"Review: {review}\nPrediction: {label} (confidence: {conf:.2f})\n")

Review: I absolutely loved this film, the acting was superb.
Prediction: Negative (confidence: 0.51)

Review: This was a complete waste of time, terrible plot.
Prediction: Negative (confidence: 0.51)

Review: An average movie, nothing special but not bad either.
Prediction: Negative (confidence: 0.51)

